# MED source-free transfer: Panasonic 25C -> CMU eVTOL core-seven

Runs `scripts/benchmarks/run_sftnn_transfer_cmu.py`: the paper-faithful MED
method (Shen et al. 2023, Algorithm 1) transferred from the Panasonic
18650PF 25C source domain to the CMU eVTOL target domain, compared against
a zero-shot (no-adaptation) baseline.

MED's transfer stage is source-free: it never sees Panasonic data/labels or
real CMU labels. Pseudo-labels come once from the frozen source model's
averaged-head output on CMU's *unlabeled* train-split missions; only the
target feature extractor is trained. Real CMU `soc_dynamic` labels are used
only afterwards, to score the held-out test-split missions -- and
`soc_dynamic` is a development dynamic-capacity reference, not measured
ground truth, so every result row is `label_is_ground_truth=false` /
`formal_benchmark_eligible=false`, matching the rest of this repo's CMU
eVTOL benchmarks.

Every result row also carries a wall-clock timing breakdown (source
pretraining, pseudo-label construction, adaptation, evaluation, and which
device it ran on) -- see Section 4.

Run cells top to bottom. Path setup happens first so the smoke test and
every later cell can just `import battery_benchmark...` or shell out to the
script directly.

## 0. Path setup

In [ ]:
# If this repo isn't already checked out in the Colab runtime, clone it here
# and adjust REPO_ROOT accordingly. Skip this cell if you're running from an
# already-mounted/cloned copy (e.g. Drive-synced local checkout).
import os
from pathlib import Path

REPO_ROOT = Path("/content/battery-soc-transfer-benchmark")
if not REPO_ROOT.exists():
    !git clone <YOUR_REPO_URL> {REPO_ROOT}
os.chdir(REPO_ROOT)
print("cwd:", Path.cwd())

In [ ]:
import sys

for path in (REPO_ROOT, REPO_ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torch
print("cuda available:", torch.cuda.is_available())

SCRIPT = REPO_ROOT / "scripts" / "benchmarks" / "run_sftnn_transfer_cmu.py"
OUTPUT_ROOT = REPO_ROOT / "outputs" / "benchmarks" / "panasonic_to_cmu_evtol" / "sftnn_med_transfer"
assert SCRIPT.is_file(), SCRIPT

## 1. Smoke test

Tiny epoch counts, one cell, both methods -- just confirms the pipeline
runs end to end (data loading, source pretraining, pseudo-label
confidence filtering, MED adaptation, evaluation, result-file writing)
before committing GPU time to a real run. A loose `--pseudo-threshold` is
used here on purpose: with only 2 pretraining epochs the source model's
predictions are too noisy for the paper-default threshold (0.08) to keep
any pseudo-labels at all -- that's expected at this epoch count, not a bug.

In [ ]:
!python {SCRIPT} \
  --cells VAH01 --methods zero_shot med --seeds 42 \
  --source-epochs 2 --transfer-epochs 2 --early-stopping-patience 1000 \
  --pseudo-threshold 1.0 \
  --output-root /tmp/sftnn_smoke_test --no-save-checkpoints --device cpu

If that printed a `med` and a `zero_shot` row with no traceback, the
pipeline is wired correctly. Now run the real sweep below.

## 2. Full run

Defaults match the paper's own MED pretraining regime (2000 source epochs)
and the paper-default confidence threshold (0.08). `--resume` makes this
safe to re-run after a Colab disconnect -- it skips any (seed, cell,
method) that already has a result JSON on disk. Add `--methods zero_shot
med dann_mmd` if you also want the DANN+MMD comparison baseline (roughly
doubles runtime, since it pretrains and adapts a second model).

Every row the script writes records wall-clock seconds for source
pretraining, pseudo-label construction, adaptation, and evaluation, plus
which device (`cpu`/`cuda`) they ran on -- see Section 4 below.

In [ ]:
!python {SCRIPT} \
  --cells VAH01 VAH17 VAH27 VAH30 VAH15 VAH12 VAH11 \
  --methods zero_shot med \
  --seeds 42 \
  --source-epochs 2000 --transfer-epochs 200 --early-stopping-patience 30 \
  --pseudo-threshold 0.08 \
  --output-root {OUTPUT_ROOT} \
  --resume

## 3. Accuracy results

Loads whatever is on disk so far under `OUTPUT_ROOT` -- safe to re-run this
section while the sweep above is still going, or after a partial run.

In [ ]:
import json
import pandas as pd


def load_sftnn_results(output_root: Path) -> pd.DataFrame:
    """No-I/O-surprises loader: reads every per-(seed, cell, method) result
    JSON directly, rather than trusting cell_metrics.csv to be fresh (it is
    rebuilt by the script on every write, but this is robust even if a run
    was interrupted mid-write)."""
    rows = []
    for result_path in sorted(output_root.glob("seed_*/*/*.json")):
        rows.append(json.loads(result_path.read_text(encoding="utf-8")))
    if not rows:
        return pd.DataFrame()
    table = pd.DataFrame(rows)
    return table.drop_duplicates(subset=["method", "cell_id", "seed"])


results = load_sftnn_results(OUTPUT_ROOT)
results

In [ ]:
# Per-cell MED vs zero-shot MAE/RMSE, averaged across seeds.
if not results.empty:
    summary = (
        results.groupby(["cell_id", "method"])[
            ["overall_mae_percent", "overall_rmse_percent"]
        ]
        .mean()
        .unstack("method")
    )
    display(summary)
else:
    print("No results yet under", OUTPUT_ROOT)

In [ ]:
import matplotlib.pyplot as plt

if not results.empty:
    pivot = results.pivot_table(
        index="cell_id", columns="method", values="overall_mae_percent", aggfunc="mean"
    )
    ax = pivot.plot(kind="bar", figsize=(10, 4))
    ax.set_ylabel("MAE (%)")
    ax.set_title("MED source-free transfer vs. zero-shot, per CMU eVTOL cell\n(dev labels -- not formal-benchmark-eligible)")
    plt.tight_layout()
    plt.show()
else:
    print("No results yet under", OUTPUT_ROOT)

## 4. Compute-time results

Every row already has a timing breakdown in seconds:

* `med_source_pretrain_seconds` / `med_source_pretrain_epochs` -- one-off
  per seed (same value repeated across every cell/method row for that
  seed, since pretraining happens once and is then reused).
* `dann_source_pretrain_seconds` / `dann_source_pretrain_epochs` -- only
  populated if `dann_mmd` was included in `--methods`.
* `pseudo_label_seconds` -- time to build the pseudo-labeled target
  dataset (MED only; confidence filtering runs per CMU mission).
* `adapt_seconds` -- time spent actually training (MED's target extractor,
  or DANN+MMD's target extractor + discriminator).
* `eval_seconds` -- time to score the held-out test-split missions.
* `device` -- `cpu` or `cuda`; seconds are meaningless without knowing
  which one produced them, so always read this alongside the timings.

`run_metadata.json`'s `this_invocation_wall_seconds` is wall-clock time for
*this specific invocation only* -- with `--resume`, anything skipped
because a result file already existed doesn't count there. The per-row
breakdown below is cumulative across every invocation that ever wrote a
result file, so it's the more reliable source for "how long did this
actually take."

In [ ]:
timing_columns = [
    "cell_id", "method", "seed", "device",
    "med_source_pretrain_seconds", "med_source_pretrain_epochs",
    "dann_source_pretrain_seconds", "dann_source_pretrain_epochs",
    "pseudo_label_seconds", "adapt_seconds", "eval_seconds",
]

if not results.empty:
    timing = results[timing_columns].copy()
    timing["row_seconds"] = timing[
        ["pseudo_label_seconds", "adapt_seconds", "eval_seconds"]
    ].sum(axis=1, skipna=True)
    display(timing)

    # One pretrain cost per seed (it happens once, not once per cell/method).
    pretrain_once = timing.drop_duplicates(subset=["seed", "device"])[
        ["seed", "device", "med_source_pretrain_seconds", "dann_source_pretrain_seconds"]
    ]
    total_pretrain = pretrain_once[
        ["med_source_pretrain_seconds", "dann_source_pretrain_seconds"]
    ].sum(skipna=True).sum()
    total_transfer_eval = timing["row_seconds"].sum()
    print(f"total pretraining time across seeds: {total_pretrain:.1f}s")
    print(f"total pseudo-label + adapt + eval time across all rows: {total_transfer_eval:.1f}s")
    print(f"grand total (pretrain once per seed + every row): {total_pretrain + total_transfer_eval:.1f}s")
else:
    print("No results yet under", OUTPUT_ROOT)

In [ ]:
# Per-cell adaptation time (MED) -- useful for estimating how long adding
# more cells or seeds will take.
if not results.empty:
    med_rows = results[results["method"] == "med"]
    if not med_rows.empty:
        ax = med_rows.set_index("cell_id")["adapt_seconds"].plot(
            kind="bar", figsize=(10, 4), color="steelblue"
        )
        ax.set_ylabel("MED adaptation time (s)")
        ax.set_title("Per-cell MED adaptation wall-clock time")
        plt.tight_layout()
        plt.show()
    else:
        print("No 'med' rows yet")
else:
    print("No results yet under", OUTPUT_ROOT)